# Demo 02 : Reducers

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · add-messages

We edit a message list. `add_messages` appends new ids, replaces an existing id and applies `RemoveMessage`.

In [2]:
from langchain.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages

history = add_messages([], [HumanMessage("Build Comet Snake with shields", id="brief"), AIMessage("Pending", id="reply")])
history = add_messages(history, [AIMessage("Coder started", id="reply")])
print("edited :", [(message.id, message.content) for message in history])
history = add_messages(history, [RemoveMessage(id="brief")])
print("removed :", [(message.id, message.content) for message in history])

class CrewState(MessagesState):
    game: str

messages = StateGraph(CrewState)
messages.add_node("reply", lambda state: {"messages": [AIMessage("Coder notified")]})
messages.add_edge(START, "reply")
messages.add_edge("reply", END)
result = messages.compile().invoke({"messages": [("user", "Build Comet Snake with shields")], "game": "CS-01"})
print("messages :", [message.content for message in result["messages"]])
print("game :", result["game"])

edited : [('brief', 'Build Comet Snake with shields'), ('reply', 'Coder started')]
removed : [('reply', 'Coder started')]
messages : ['Build Comet Snake with shields', 'Coder notified']
game : CS-01


## b · custom-reducers

We start with a small sequential example : `bugs` has a reducer and grows, `status` has none and is overwritten. We then merge bug reports with a custom reducer that removes duplicates and sorts the result.

In [3]:
import operator
from typing import Annotated, TypedDict

class Collect(TypedDict):
    bugs: Annotated[list[str], operator.add]
    status: str

collected = StateGraph(Collect)
collected.add_node("rule_tester", lambda state: {"bugs": ["self-hit"], "status": "rules done"})
collected.add_node("bot_tester", lambda state: {"bugs": ["bot-score"], "status": "bot done"})
collected.add_edge(START, "rule_tester")
collected.add_edge("rule_tester", "bot_tester")
collected.add_edge("bot_tester", END)
print("sequential :", collected.compile().invoke({"bugs": [], "status": ""}))

def merge_bugs(current: list[str], incoming: list[str]) -> list[str]:
    return sorted(set(current) | set(incoming))

class Bugs(TypedDict):
    bugs: Annotated[list[str], merge_bugs]

bugs = StateGraph(Bugs)
bugs.add_node("tester", lambda state: {"bugs": ["self-hit", "self-hit"]})
bugs.add_edge(START, "tester")
bugs.add_edge("tester", END)
print("merged :", bugs.compile().invoke({"bugs": ["more-comets"]}))

sequential : {'bugs': ['self-hit', 'bot-score'], 'status': 'bot done'}
merged : {'bugs': ['more-comets', 'self-hit']}


## c · overwriting

A new build starts with an empty bug list. We replace a reduced value with Overwrite. An ordinary empty list would pass through the reducer and keep the existing bugs.

In [4]:
from langgraph.types import Overwrite

reset = StateGraph(Bugs)
reset.add_node("clear", lambda state: {"bugs": Overwrite(value=[])})
reset.add_edge(START, "clear")
reset.add_edge("clear", END)
print("ordinary empty update :", merge_bugs(["more-comets"], []))
print("overwrite :", reset.compile().invoke({"bugs": ["more-comets"]}))

ordinary empty update : ['more-comets']
overwrite : {'bugs': []}


An `Overwrite` and an ordinary write to the same key can meet in one superstep. Branch `a` replaces the bug list while branch `b` adds one.

In [5]:
race = StateGraph(Bugs)
race.add_node("a", lambda state: {"bugs": Overwrite(["self-hit"])})
race.add_node("b", lambda state: {"bugs": ["bot-score"]})
race.add_edge(START, "a")
race.add_edge(START, "b")
race.add_edge(["a", "b"], END)
print("overwrite and write :", race.compile().invoke({"bugs": ["more-comets"]}))

overwrite and write : {'bugs': ['self-hit']}


The ordinary write from `b` is gone, and no error is raised. Two `Overwrite` values for one key in one step do raise `InvalidUpdateError`.

## d · concurrent-updates

We run two parallel testers that write one unreduced key. We catch InvalidUpdateError, then keep both results with a reducer.

In [6]:
from langgraph.errors import InvalidUpdateError

class NoReducer(TypedDict):
    bug: str

parallel = StateGraph(NoReducer)
parallel.add_node("rule_tester", lambda state: {"bug": "self-hit"})
parallel.add_node("bot_tester", lambda state: {"bug": "bot-score"})
parallel.add_edge(START, "rule_tester")
parallel.add_edge(START, "bot_tester")
parallel.add_edge("rule_tester", END)
parallel.add_edge("bot_tester", END)
try:
    parallel.compile().invoke({"bug": ""})
except InvalidUpdateError as error:
    print("InvalidUpdateError :", str(error).splitlines()[0])

fixed = StateGraph(Bugs)
fixed.add_node("rule_tester", lambda state: {"bugs": ["self-hit"]})
fixed.add_node("bot_tester", lambda state: {"bugs": ["bot-score"]})
fixed.add_edge(START, "rule_tester")
fixed.add_edge(START, "bot_tester")
fixed.add_edge("rule_tester", END)
fixed.add_edge("bot_tester", END)
print("fixed :", fixed.compile().invoke({"bugs": []}))

InvalidUpdateError : At key 'bug': Can receive only one value per step. Use an Annotated key to handle multiple values.
fixed : {'bugs': ['bot-score', 'self-hit']}


The same two testers on a real game. We break Comet Snake on purpose : the fifth star pushes an empty comet. The rule tests and the bot run the file through QuickJS at the same time and write to one bug list.

In [7]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]
import importlib.util
from pathlib import Path

kit = Path(data_folder("game-studio/kit"))
spec = importlib.util.spec_from_file_location("harness", kit / "harness.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)

TEMPLATE = (kit / "template.html").read_text(encoding="utf-8")
RULES = (kit / "rules.md").read_text(encoding="utf-8")
TWISTS = json.loads((kit / "twists.json").read_text(encoding="utf-8"))
BRIEFS = json.loads((kit / "briefs.json").read_text(encoding="utf-8"))

In [8]:
game = Path(data("game-studio/games/comet-snake.html")).read_text(encoding="utf-8")
broken = game.replace("if (state.comets.length < 6) spawnComet(state);", "state.comets.push(state.comets[9]);")

class TestBoard(TypedDict):
    bugs: Annotated[list[str], operator.add]

def rule_tester(board: TestBoard):
    report = harness.run_tests(broken, bot=False)
    return {"bugs": [f"rule tester, {item['test']} : {item['message']}" for item in report["failures"]][:2]}

def bot_tester(board: TestBoard):
    report = harness.run_tests(broken)
    return {"bugs": [f"bot tester, {item['test']} : {item['message']}" for item in report["failures"] if item["test"].startswith("bot")]}

side_by_side = StateGraph(TestBoard)
side_by_side.add_node("rule_tester", rule_tester)
side_by_side.add_node("bot_tester", bot_tester)
side_by_side.add_edge(START, "rule_tester")
side_by_side.add_edge(START, "bot_tester")
side_by_side.add_edge("rule_tester", END)
side_by_side.add_edge("bot_tester", END)
for bug in side_by_side.compile().invoke({"bugs": []})["bugs"]:
    print(bug.split(", at same")[0])

bot tester, bot-plays : crash, JSError: TypeError: cannot read property 'x' of undefined
rule tester, speed-fifth : crash, JSError: TypeError: cannot read property 'x' of undefined
rule tester, speed-floor : crash, JSError: TypeError: cannot read property 'x' of undefined


## Things to try

Optional : we can change a brief, a thread id or a returned update and compare the saved state.